# 07 AARRR feature mapping 260513

Step 07 gate notebook. This notebook maps existing source columns and step 06 conservative features to AARRR stages. It is conceptual and policy-oriented only. No modeling, prediction, SHAP, Optuna, statistical testing, plotting, row exclusion, or feature engineering is performed.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import zipfile

import numpy as np
import pandas as pd

STEP_NAME = '07_AARRR_feature_mapping_260513'
EXPECTED_REPO_ROOTS = ['C:/Code/ott-churn-prediction', 'C:\\Code\\ott-churn-prediction']
EXPECTED_RAW_ROWS = 23343
EXPECTED_RAW_COLS = 91
EXPECTED_MAIN_ROWS = 23079
EXPECTED_CONSERVATIVE_FEATURES = 22

def is_inside(child, parent):
    try:
        Path(child).resolve().relative_to(Path(parent).resolve())
        return True
    except ValueError:
        return False

def write_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8-sig')

def yn(value):
    return 'yes' if bool(value) else 'no'

def safe_join(values):
    vals = [str(v) for v in values if pd.notna(v) and str(v) != '']
    return ';'.join(vals)

actual_repo_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
repo_root_match = actual_repo_root in EXPECTED_REPO_ROOTS
print('actual_repo_root:', actual_repo_root)
print('repo_root_match:', repo_root_match)
if not repo_root_match:
    raise SystemExit('STOP: repo root mismatch. No files were written.')

ROOT = Path(actual_repo_root).resolve()
PARK = ROOT / 'park.ingyeom'
SOURCE = PARK / 'data' / '(광일)Membership_v2_with_derived_features.csv'
NOTE = PARK / 'note.md'
NOTEBOOK_PATH = PARK / 'notebook' / STEP_NAME / f'{STEP_NAME}.ipynb'
OUTPUT_BASE = PARK / 'reports' / 'audits' / STEP_NAME
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP_NAME}_review_package.zip'

previous_final_checks = [
    PARK / 'reports' / 'audits' / '01_data_contract_260513' / '01_final_checks.csv',
    PARK / 'reports' / 'audits' / '02_target_score_orientation_260513' / '02_final_checks.csv',
    PARK / 'reports' / 'audits' / '03_observation_window_policy_260513' / '03_final_checks.csv',
    PARK / 'reports' / 'audits' / '04_promotion_split_260513' / '04_final_checks.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_final_checks.csv',
    PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_final_checks.csv',
]
files_05b = [
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_canonical_column_role_dictionary.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_canonical_timing_audit.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_canonical_recommended_feature_set_contracts.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_conservative_safe_candidate_columns.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_review_required_columns.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_forbidden_drop_columns.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_downstream_handoff_policy.csv',
]
files_06 = [
    PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_primary_main_cohort_index.csv',
    PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_primary_main_cohort_conservative_features.csv',
    PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_feature_policy_from_05b.csv',
    PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_common_preprocessing_policy.csv',
    PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_downstream_handoff_for_11_baseline_ladder.csv',
    PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_open_risks_for_next_steps.csv',
]

if OUTPUT_BASE.exists() and any(OUTPUT_BASE.iterdir()):
    OUTPUT_DIR = OUTPUT_BASE / ('run_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
else:
    OUTPUT_DIR = OUTPUT_BASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

preflight = {
    'expected_repo_root': 'C:/Code/ott-churn-prediction or C:\\Code\\ott-churn-prediction',
    'actual_repo_root': actual_repo_root,
    'repo_root_match': repo_root_match,
    'source_file_exists': SOURCE.exists(),
    'all_required_previous_final_checks_exist': all(p.exists() for p in previous_final_checks),
    'all_required_05b_files_exist': all(p.exists() for p in files_05b),
    'all_required_06_files_exist': all(p.exists() for p in files_06),
    'note_md_exists': NOTE.exists(),
    'source_file_inside_park_ingyeom': is_inside(SOURCE, PARK),
    'output_folder_inside_park_ingyeom': is_inside(OUTPUT_DIR, PARK),
    'notebook_inside_park_ingyeom': is_inside(NOTEBOOK_PATH, PARK),
    'zip_folder_inside_park_ingyeom': is_inside(ZIP_DIR, PARK),
}
preflight['can_proceed'] = all(bool(v) for k, v in preflight.items() if k not in ['expected_repo_root', 'actual_repo_root'])
write_csv(pd.DataFrame([{'check_name': k, 'value': v} for k, v in preflight.items()]), OUTPUT_DIR / '07_preflight_input_validation.csv')

if not preflight['can_proceed']:
    missing = [str(p.relative_to(PARK)) for p in [SOURCE, NOTE] + previous_final_checks + files_05b + files_06 if not p.exists()]
    readme = ['# 07 AARRR feature mapping', '', 'This run stopped at preflight validation.', '', 'No downstream mapping outputs were created.', '', 'Missing or failed inputs:']
    readme.extend([f'- {m}' for m in missing] or ['- See 07_preflight_input_validation.csv'])
    (OUTPUT_DIR / 'README.md').write_text('\n'.join(readme) + '\n', encoding='utf-8')
    raise SystemExit('STOP: preflight failed. Only validation and README were written.')

source_stat_before = SOURCE.stat()
df = pd.read_csv(SOURCE, encoding='utf-8-sig')
dict_df = pd.read_csv(files_05b[0], encoding='utf-8-sig')
timing_df = pd.read_csv(files_05b[1], encoding='utf-8-sig')
safe_df = pd.read_csv(files_05b[3], encoding='utf-8-sig')
review_df = pd.read_csv(files_05b[4], encoding='utf-8-sig')
forbidden_df = pd.read_csv(files_05b[5], encoding='utf-8-sig')
cohort_index = pd.read_csv(files_06[0], encoding='utf-8-sig')
conservative_table = pd.read_csv(files_06[1], encoding='utf-8-sig')
policy06 = pd.read_csv(files_06[2], encoding='utf-8-sig')

source_cols = list(df.columns)
safe_set = set(safe_df['column_name'].astype(str))
review_set = set(review_df['column_name'].astype(str))
forbidden_set = set(forbidden_df['column_name'].astype(str))
metadata_flag_cols = {'source_row_number', 'USER_KEY', 'is_promotion', 'is_repurchase', 'duration_days'} | {c for c in conservative_table.columns if c.startswith('flag_')}
conservative_feature_cols = [c for c in conservative_table.columns if c not in metadata_flag_cols]

def status(condition):
    return 'PASS' if bool(condition) else 'MISMATCH'

consistency_rows = [
    {'check_name': 'raw source row count', 'actual_value': len(df), 'expected_value': EXPECTED_RAW_ROWS, 'status': status(len(df) == EXPECTED_RAW_ROWS), 'detail': ''},
    {'check_name': 'raw source column count', 'actual_value': len(df.columns), 'expected_value': EXPECTED_RAW_COLS, 'status': status(len(df.columns) == EXPECTED_RAW_COLS), 'detail': ''},
    {'check_name': 'primary main cohort row count', 'actual_value': len(cohort_index), 'expected_value': EXPECTED_MAIN_ROWS, 'status': status(len(cohort_index) == EXPECTED_MAIN_ROWS), 'detail': ''},
    {'check_name': 'conservative feature table row count', 'actual_value': len(conservative_table), 'expected_value': EXPECTED_MAIN_ROWS, 'status': status(len(conservative_table) == EXPECTED_MAIN_ROWS), 'detail': ''},
    {'check_name': 'conservative feature count from 06', 'actual_value': len(conservative_feature_cols), 'expected_value': EXPECTED_CONSERVATIVE_FEATURES, 'status': status(len(conservative_feature_cols) == EXPECTED_CONSERVATIVE_FEATURES), 'detail': safe_join(conservative_feature_cols)},
    {'check_name': 'source target column exists', 'actual_value': yn('is_repurchase' in df.columns), 'expected_value': 'yes', 'status': status('is_repurchase' in df.columns), 'detail': 'is_repurchase'},
    {'check_name': 'source split column exists', 'actual_value': yn('is_promotion' in df.columns), 'expected_value': 'yes', 'status': status('is_promotion' in df.columns), 'detail': 'is_promotion'},
    {'check_name': 'USER_KEY exists', 'actual_value': yn('USER_KEY' in df.columns), 'expected_value': 'yes', 'status': status('USER_KEY' in df.columns), 'detail': 'USER_KEY'},
    {'check_name': '05b canonical dictionary has 91 rows', 'actual_value': len(dict_df), 'expected_value': 91, 'status': status(len(dict_df) == 91), 'detail': ''},
    {'check_name': '05b timing audit has 91 rows', 'actual_value': len(timing_df), 'expected_value': 91, 'status': status(len(timing_df) == 91), 'detail': ''},
    {'check_name': '06 conservative table has only safe candidate feature columns plus required metadata/flags', 'actual_value': yn(set(conservative_feature_cols).issubset(safe_set)), 'expected_value': 'yes', 'status': status(set(conservative_feature_cols).issubset(safe_set)), 'detail': safe_join(sorted(set(conservative_feature_cols) - safe_set))},
    {'check_name': 'no review columns in conservative feature columns', 'actual_value': len(set(conservative_feature_cols) & review_set), 'expected_value': 0, 'status': status(len(set(conservative_feature_cols) & review_set) == 0), 'detail': safe_join(sorted(set(conservative_feature_cols) & review_set))},
    {'check_name': 'no forbidden/drop columns in conservative feature columns', 'actual_value': len(set(conservative_feature_cols) & forbidden_set), 'expected_value': 0, 'status': status(len(set(conservative_feature_cols) & forbidden_set) == 0), 'detail': safe_join(sorted(set(conservative_feature_cols) & forbidden_set))},
]
write_csv(pd.DataFrame(consistency_rows), OUTPUT_DIR / '07_source_cohort_consistency_check.csv')

stage_contract = pd.DataFrame([
    {'AARRR_stage': 'Acquisition', 'project_specific_meaning': 'promotion/non-promotion inflow distinction', 'directly_measurable': 'yes', 'primary_field': 'is_promotion', 'candidate_conservative_features': '', 'caveat': 'Descriptive split only, not causal promotion effect.'},
    {'AARRR_stage': 'Activation', 'project_specific_meaning': 'whether the subscription event converts into early viewing behavior', 'directly_measurable': 'yes', 'primary_field': 'week1 and cold-start features', 'candidate_conservative_features': safe_join([c for c in conservative_feature_cols if ('w1' in c or 'cold_start' in c)]), 'caveat': 'No direct satisfaction data.'},
    {'AARRR_stage': 'Retention', 'project_specific_meaning': 'whether viewing behavior is maintained or decays from week1 to week3', 'directly_measurable': 'yes', 'primary_field': 'w1/w2/w3 retention and change features', 'candidate_conservative_features': safe_join([c for c in conservative_feature_cols if c not in [x for x in conservative_feature_cols if ('w1' in x and 'w2' not in x and 'w3' not in x) or 'cold_start' in x]]), 'caveat': 'Observation ends at day20; response period begins day21.'},
    {'AARRR_stage': 'Revenue proxy', 'project_specific_meaning': 'next-month repurchase / continuation proxy', 'directly_measurable': 'yes as target proxy', 'primary_field': 'is_repurchase', 'candidate_conservative_features': '', 'caveat': 'Not actual revenue amount; do not call it exact monetary revenue.'},
    {'AARRR_stage': 'Referral', 'project_specific_meaning': 'recommendation/sharing/invite behavior', 'directly_measurable': 'no', 'primary_field': 'none', 'candidate_conservative_features': '', 'caveat': 'No referral log, invite log, sharing log, or campaign response log. Treat only as post-analysis growth experiment hypothesis.'},
])
write_csv(stage_contract, OUTPUT_DIR / '07_AARRR_stage_contract.csv')

def conceptual_stage(col, family, role):
    c = str(col)
    f = str(family)
    r = str(role)
    if c == 'is_promotion':
        return 'acquisition', ''
    if c == 'is_repurchase':
        return 'revenue_proxy', ''
    if c == 'USER_KEY' or r in ['id', 'date_or_time_anchor', 'target', 'split']:
        return 'metadata_id_split_target', ''
    if 'referral' in c.lower() or 'invite' in c.lower() or 'recommend' in c.lower():
        return 'referral_proposal_only', ''
    if any(k in c for k in ['w2', 'w3', 'retention', 'diff_between', 'gap']) or 'retention' in f:
        return 'retention', ''
    if any(k in c for k in ['w1', 'cold_start', 'reg_hour', 'reg_is_weekend', 'payment_is', 'is_only_w1']) or 'activation' in f:
        return 'activation', ''
    if any(k in c for k in ['price', 'premium', 'standard', 'product_code', 'billing', 'max_screen']):
        return 'revenue_proxy', 'acquisition'
    if any(k in c for k in ['genre', 'ratio', 'movie', 'watch', 'active', 'recency', 'ott_release']):
        return 'retention', 'activation'
    if any(k in c for k in ['age', 'gender', 'female', 'male', 'verified']):
        return 'acquisition', 'activation'
    return 'unknown_review', ''

def mapping_for_row(row):
    col = str(row['column_name'])
    role = str(row.get('patched_primary_role', ''))
    family = str(row.get('patched_feature_family', ''))
    stage, secondary = conceptual_stage(col, family, role)
    in_cons = col in conservative_feature_cols
    if col == 'USER_KEY':
        return stage, secondary, 'directly_observed', 'metadata_only', 'Identifier metadata only, not AARRR behavior.', 'Use only as group key or lineage metadata.', 'exclude'
    if col == 'is_repurchase':
        return 'revenue_proxy', '', 'target_proxy', 'target_only', 'Target proxy for next-month repurchase, not actual revenue amount.', 'Never use as feature.', 'target_only'
    if col == 'is_promotion':
        return 'acquisition', '', 'directly_observed', 'split_only', 'Promotion/non-promotion split metadata.', 'Descriptive only; not causal effect.', 'split_only'
    if col in forbidden_set:
        return stage if stage != 'unknown_review' else 'exclusion_or_review', secondary, 'forbidden', 'forbidden_or_drop', '05b marks this as forbidden/drop or non-feature control.', 'Exclude from standard modeling.', 'exclude'
    if col in review_set:
        return stage, secondary, 'review_required', 'review_only', 'Conceptually mappable but 05b requires timing or semantic confirmation.', 'Do not approve for standard modeling here.', 'review_before_modeling'
    if in_cons:
        return stage, secondary, 'proxy_observed', 'standard_conservative_usable', '05b conservative safe candidate and present in 06 conservative table.', 'Proxy behavior feature within day0 to day20 window.', 'standard_baseline_candidate'
    return stage, secondary, 'review_required', 'review_only', 'Not in 06 conservative table under current conservative policy.', 'Keep out until explicitly resolved.', 'review_before_modeling'

policy06_small = policy06[['column_name', 'status_for_06_primary_main_cohort_table']].copy()
all_map_base = dict_df.merge(policy06_small, on='column_name', how='left')
all_rows = []
for _, row in all_map_base.iterrows():
    stage, secondary, measurement, std_status, reason, caveat, downstream = mapping_for_row(row)
    all_rows.append({
        'column_name': row['column_name'],
        'patched_primary_role from 05b': row.get('patched_primary_role', ''),
        'patched_feature_family from 05b': row.get('patched_feature_family', ''),
        'patched_timing_family from 05b': row.get('patched_timing_family', ''),
        '05b overall allowed status': row.get('patched_allowed_for_overall_model_candidate', ''),
        '05b groupwise allowed status': row.get('patched_allowed_for_groupwise_model_candidate', ''),
        '06 status_for_06_primary_main_cohort_table if available': row.get('status_for_06_primary_main_cohort_table', ''),
        'in_06_conservative_feature_table': yn(row['column_name'] in conservative_feature_cols),
        'AARRR_stage_primary': stage,
        'AARRR_stage_secondary': secondary,
        'measurement_status': measurement,
        'standard_analysis_status': std_status,
        'reason': reason,
        'caveat': caveat,
        'recommended_downstream_use': downstream,
    })
all_mapping = pd.DataFrame(all_rows)
write_csv(all_mapping, OUTPUT_DIR / '07_AARRR_feature_mapping_all_columns.csv')

def ladder_family(col, stage):
    if stage == 'activation':
        return 'L1_activation_safe_window'
    if stage == 'retention' and ('w2' in col or 'retention_w2' in col):
        return 'L2_retention_w2_safe_window'
    if stage == 'retention' and ('w3' in col or 'retention_w3' in col):
        return 'L3_retention_w3_safe_window'
    if stage == 'retention':
        return 'L4_safe_behavioral_combined'
    return 'L0_conservative_minimal_baseline'

cons_rows = []
for col in conservative_feature_cols:
    drow = dict_df.loc[dict_df['column_name'] == col].iloc[0]
    stage, _ = conceptual_stage(col, drow.get('patched_feature_family', ''), drow.get('patched_primary_role', ''))
    cons_rows.append({
        'column_name': col,
        'AARRR_stage_primary': stage,
        'feature_family': drow.get('patched_feature_family', ''),
        'timing_family': drow.get('patched_timing_family', ''),
        'why_usable_conservatively': 'Present in 05b conservative safe candidate list and in 06 conservative feature table.',
        'modeling_ladder_family_suggestion': ladder_family(col, stage),
        'expected_interpretation_if_important_later': 'Interpret as descriptive day0-20 behavior association with repurchase target, not causal effect.',
        'caveat': 'Proxy feature only; future modeling must keep target, split, and group rules.'
    })
cons_mapping = pd.DataFrame(cons_rows)
summary_rows = []
for stage, count in cons_mapping['AARRR_stage_primary'].value_counts().sort_index().items():
    summary_rows.append({'column_name': f'__summary_stage_count__{stage}', 'AARRR_stage_primary': stage, 'feature_family': '', 'timing_family': '', 'why_usable_conservatively': f'count={count}', 'modeling_ladder_family_suggestion': '', 'expected_interpretation_if_important_later': '', 'caveat': ''})
for family, count in cons_mapping['feature_family'].value_counts().sort_index().items():
    summary_rows.append({'column_name': f'__summary_family_count__{family}', 'AARRR_stage_primary': '', 'feature_family': family, 'timing_family': '', 'why_usable_conservatively': f'count={count}', 'modeling_ladder_family_suggestion': '', 'expected_interpretation_if_important_later': '', 'caveat': ''})
summary_rows.extend([
    {'column_name': '__summary_acquisition_feature_candidates__', 'AARRR_stage_primary': 'acquisition', 'feature_family': '', 'timing_family': '', 'why_usable_conservatively': 'Acquisition has only split metadata is_promotion, not ordinary conservative feature candidates.', 'modeling_ladder_family_suggestion': '', 'expected_interpretation_if_important_later': '', 'caveat': 'Do not use is_promotion as groupwise model feature.'},
    {'column_name': '__summary_revenue_feature_candidates__', 'AARRR_stage_primary': 'revenue_proxy', 'feature_family': '', 'timing_family': '', 'why_usable_conservatively': 'Revenue has target proxy is_repurchase only, not a feature.', 'modeling_ladder_family_suggestion': '', 'expected_interpretation_if_important_later': '', 'caveat': 'Do not use target as feature.'},
    {'column_name': '__summary_referral_feature_candidates__', 'AARRR_stage_primary': 'referral_proposal_only', 'feature_family': '', 'timing_family': '', 'why_usable_conservatively': 'Referral has no observed feature in current data.', 'modeling_ladder_family_suggestion': '', 'expected_interpretation_if_important_later': '', 'caveat': 'Future experiment proposal only.'},
])
write_csv(pd.concat([cons_mapping, pd.DataFrame(summary_rows)], ignore_index=True), OUTPUT_DIR / '07_AARRR_mapping_conservative_features.csv')

review_rows = []
for _, row in review_df.iterrows():
    col = str(row['column_name'])
    stage, _ = conceptual_stage(col, row.get('patched_feature_family', ''), row.get('patched_primary_role', ''))
    review_rows.append({
        'column_name': col,
        'conceptual_AARRR_stage': stage,
        'why_review_required': row.get('patched_reason', row.get('reason', '05b review-required column.')),
        'what_confirmation_is_needed': row.get('patched_future_step_to_resolve', row.get('future_step_to_resolve', 'explicit semantic/timing confirmation')),
        'possible_use_after_resolution': 'May be used in review-resolved baseline or sensitivity experiment only after explicit approval.',
        'risk_if_used_now': 'Could introduce timing leakage, semantic ambiguity, or unsupported business interpretation.'
    })
write_csv(pd.DataFrame(review_rows), OUTPUT_DIR / '07_AARRR_mapping_review_required_columns.csv')

def cols_for_stage(stage_name):
    return safe_join(cons_mapping.loc[cons_mapping['AARRR_stage_primary'] == stage_name, 'column_name'])

limitations = pd.DataFrame([
    {'stage': 'Acquisition', 'can_measure_directly_from_current_data': 'yes', 'measurement_type': 'direct', 'available_columns_or_features': 'is_promotion', 'unavailable_data': 'true acquisition channel, campaign exposure, randomized treatment', 'safe_claim': 'is_promotion directly separates promotion and non-promotion rows descriptively.', 'unsafe_claim': 'is_promotion proves acquisition causal effect.', 'downstream_needed_check': 'Compare distributions without causal language.'},
    {'stage': 'Activation', 'can_measure_directly_from_current_data': 'partial', 'measurement_type': 'proxy', 'available_columns_or_features': cols_for_stage('activation'), 'unavailable_data': 'satisfaction, onboarding completion, explicit intent', 'safe_claim': 'Early viewing/cold-start columns proxy activation within the observation window.', 'unsafe_claim': 'Activation quality is fully measured.', 'downstream_needed_check': 'EDA distribution by promotion and target.'},
    {'stage': 'Retention', 'can_measure_directly_from_current_data': 'partial', 'measurement_type': 'proxy', 'available_columns_or_features': cols_for_stage('retention'), 'unavailable_data': 'post-day21 behavior as feature, long-term retention', 'safe_claim': 'Week-to-week behavior within day0-20 proxies retention/decay.', 'unsafe_claim': 'Full long-term retention is validated.', 'downstream_needed_check': 'Keep response-period behavior forbidden.'},
    {'stage': 'Revenue proxy', 'can_measure_directly_from_current_data': 'partial', 'measurement_type': 'target_proxy', 'available_columns_or_features': 'is_repurchase', 'unavailable_data': 'actual revenue amount, ARPU, margin, payment amount after scoring', 'safe_claim': 'is_repurchase is a next-month continuation/revenue proxy.', 'unsafe_claim': 'is_repurchase is actual revenue amount.', 'downstream_needed_check': 'Keep as target only, not feature.'},
    {'stage': 'Referral', 'can_measure_directly_from_current_data': 'no', 'measurement_type': 'proposal_only', 'available_columns_or_features': '', 'unavailable_data': 'referral log, invite log, sharing log, campaign response log', 'safe_claim': 'Referral is a future experiment proposal only.', 'unsafe_claim': 'Referral performance was analyzed from this dataset.', 'downstream_needed_check': 'Requires future experiment data.'},
])
write_csv(limitations, OUTPUT_DIR / '07_AARRR_measurement_limitations.csv')

business_questions = pd.DataFrame([
    {'AARRR_stage': 'Acquisition', 'business_question': 'Are promotion rows and non-promotion rows behaviorally different?', 'data_question': 'How do is_promotion groups differ descriptively?', 'candidate_columns_conservative': 'is_promotion as split metadata', 'candidate_columns_review': 'membership/context columns after review', 'expected_next_analysis_step': '08 promotion vs non-promotion EDA', 'safe_interpretation': 'Descriptive group difference.', 'unsafe_interpretation': 'Promotion caused the difference.'},
    {'AARRR_stage': 'Activation', 'business_question': 'Do early viewing and cold-start signals differ by promotion and repurchase?', 'data_question': 'How do week1/cold-start safe features vary by split and target?', 'candidate_columns_conservative': cols_for_stage('activation'), 'candidate_columns_review': safe_join([r['column_name'] for r in review_rows if r['conceptual_AARRR_stage'] == 'activation']), 'expected_next_analysis_step': '08 EDA, no tests in this step', 'safe_interpretation': 'Early behavior proxy patterns.', 'unsafe_interpretation': 'User satisfaction is proven.'},
    {'AARRR_stage': 'Retention', 'business_question': 'Does week-to-week viewing decline separate repurchase vs non-repurchase rows?', 'data_question': 'How do retention/diff features vary by split and target?', 'candidate_columns_conservative': cols_for_stage('retention'), 'candidate_columns_review': safe_join([r['column_name'] for r in review_rows if r['conceptual_AARRR_stage'] == 'retention']), 'expected_next_analysis_step': '08/09 EDA planning', 'safe_interpretation': 'Day0-20 retention proxy pattern.', 'unsafe_interpretation': 'Long-term retention is fully validated.'},
    {'AARRR_stage': 'Revenue proxy', 'business_question': 'Which groups have lower is_repurchase rates?', 'data_question': 'How does target rate differ descriptively across groups?', 'candidate_columns_conservative': 'is_repurchase as target proxy only', 'candidate_columns_review': 'price/product after review only', 'expected_next_analysis_step': 'Target-rate EDA without causal claims', 'safe_interpretation': 'Repurchase proxy rate differs descriptively.', 'unsafe_interpretation': 'Actual revenue amount was measured.'},
    {'AARRR_stage': 'Referral', 'business_question': 'Which high-activation, low-risk rows could become referral experiment candidates later?', 'data_question': 'Not answerable directly now; requires future experiment after modeling.', 'candidate_columns_conservative': '', 'candidate_columns_review': '', 'expected_next_analysis_step': 'Future A/B test proposal only', 'safe_interpretation': 'Referral is a hypothesis for later experiment design.', 'unsafe_interpretation': 'Referral outcome was observed.'},
])
write_csv(business_questions, OUTPUT_DIR / '07_AARRR_business_questions.csv')

eda_plan = pd.DataFrame([
    {'planned_eda_name': 'promotion_vs_nonpromotion_basic_comparison', 'AARRR_stage': 'Acquisition', 'required_input_table': '06_primary_main_cohort_conservative_features.csv', 'grouping variables': 'is_promotion', 'metrics': 'row count, target rate, conservative feature summaries', 'filters': 'primary main cohort only', 'why_needed': 'Establish descriptive promotion split differences.', 'expected_output_later': 'Tables and plots in step 08.', 'caution': 'No causal effect claim.'},
    {'planned_eda_name': 'promotion_x_repurchase_2x2_comparison', 'AARRR_stage': 'Acquisition / Revenue proxy', 'required_input_table': '06_primary_main_cohort_conservative_features.csv', 'grouping variables': 'is_promotion, is_repurchase', 'metrics': '2x2 count and rate', 'filters': 'primary main cohort only', 'why_needed': 'Check split by target orientation.', 'expected_output_later': '2x2 table.', 'caution': 'is_repurchase is target proxy only.'},
    {'planned_eda_name': 'activation_feature_distribution_by_promotion_and_target', 'AARRR_stage': 'Activation', 'required_input_table': '06_primary_main_cohort_conservative_features.csv', 'grouping variables': 'is_promotion, is_repurchase', 'metrics': cols_for_stage('activation'), 'filters': 'safe activation features only', 'why_needed': 'Understand early behavior proxies.', 'expected_output_later': 'Distribution summaries and plots.', 'caution': 'No statistical testing in step 07.'},
    {'planned_eda_name': 'retention_decay_feature_distribution_by_promotion_and_target', 'AARRR_stage': 'Retention', 'required_input_table': '06_primary_main_cohort_conservative_features.csv', 'grouping variables': 'is_promotion, is_repurchase', 'metrics': cols_for_stage('retention'), 'filters': 'safe retention features only', 'why_needed': 'Understand week-to-week behavior decay.', 'expected_output_later': 'Distribution summaries and plots.', 'caution': 'Observation window ends at day20.'},
    {'planned_eda_name': 'conservative_feature_summary_by_AARRR_stage', 'AARRR_stage': 'Activation / Retention', 'required_input_table': '07_AARRR_mapping_conservative_features.csv', 'grouping variables': 'AARRR_stage_primary', 'metrics': 'feature counts and family counts', 'filters': 'exclude summary rows when analyzing raw features', 'why_needed': 'Plan baseline ladder families.', 'expected_output_later': 'Feature inventory table.', 'caution': 'Mapping does not validate predictive value.'},
    {'planned_eda_name': 'duration_lt21_anomaly_reference_check', 'AARRR_stage': 'Policy / Retention', 'required_input_table': '06 anomaly/reference outputs', 'grouping variables': 'duration flags, is_promotion, is_repurchase', 'metrics': 'row count and rates', 'filters': 'reference only', 'why_needed': 'Carry anomaly context without changing main cohort.', 'expected_output_later': 'Anomaly note.', 'caution': 'Do not add rows back without sensitivity design.'},
    {'planned_eda_name': 'cross_promotion_USER_KEY_overlap_caution_check', 'AARRR_stage': 'Acquisition', 'required_input_table': '06_primary_main_cohort_index.csv', 'grouping variables': 'flag_cross_promotion_USER_KEY_overlap', 'metrics': 'row count and target rate', 'filters': 'primary main cohort only', 'why_needed': 'Avoid unique-user promotion language.', 'expected_output_later': 'Caution table.', 'caution': 'Row-level/subscription-event-level wording only.'},
    {'planned_eda_name': 'review_columns_resolution_plan_if_needed', 'AARRR_stage': 'All', 'required_input_table': '05b review required columns', 'grouping variables': 'conceptual_AARRR_stage', 'metrics': 'review count and required confirmation', 'filters': 'review columns only', 'why_needed': 'Decide later sensitivity or review-resolved baseline.', 'expected_output_later': 'Resolution checklist.', 'caution': 'Do not promote review columns in step 07.'},
])
write_csv(eda_plan, OUTPUT_DIR / '07_AARRR_to_EDA_plan.csv')

excluded_membership = safe_join([c for c in review_set if c in ['product_code','price','billing_method','max_screen','is_standard','is_premium','age','age_group','gender','is_female','is_male','is_user_verified','payment_device']])
excluded_content = safe_join([c for c in review_set if any(k in c for k in ['genre','ratio','movie','ott_release'])])
ladder_rows = [
    {'ladder_step': '1', 'proposed_name': 'L0_conservative_minimal_baseline', 'AARRR_stage': 'Activation/Retention', 'feature_family': 'safe behavior only', 'candidate_columns_from_conservative_table': safe_join(conservative_feature_cols[:3]), 'excluded_review_columns': excluded_membership, 'reason_for_exclusion': 'No strict-safe membership/context feature is available under current 05b conservative policy.', 'caveat': 'Do not label review membership-only as safe L0.', 'recommended_status': 'allowed_conservative_start'},
    {'ladder_step': '2', 'proposed_name': 'L1_activation_safe_window', 'AARRR_stage': 'Activation', 'feature_family': 'activation_onboarding / week1', 'candidate_columns_from_conservative_table': cols_for_stage('activation'), 'excluded_review_columns': excluded_membership, 'reason_for_exclusion': 'Membership/context needs semantic and timing confirmation.', 'caveat': 'Behavior proxy, not satisfaction.', 'recommended_status': 'allowed'},
    {'ladder_step': '3', 'proposed_name': 'L2_retention_w2_safe_window', 'AARRR_stage': 'Retention', 'feature_family': 'week2 retention', 'candidate_columns_from_conservative_table': safe_join([c for c in conservative_feature_cols if 'w2' in c or 'retention_w2' in c]), 'excluded_review_columns': safe_join([c for c in review_set if c in ['total_watch_count','total_watch_time(min)','recency']]), 'reason_for_exclusion': 'Total/all-period and recency timing unresolved.', 'caveat': 'Use only safe window columns.', 'recommended_status': 'allowed'},
    {'ladder_step': '4', 'proposed_name': 'L3_retention_w3_safe_window', 'AARRR_stage': 'Retention', 'feature_family': 'week3 retention', 'candidate_columns_from_conservative_table': safe_join([c for c in conservative_feature_cols if 'w3' in c or 'retention_w3' in c]), 'excluded_review_columns': safe_join([c for c in review_set if c in ['total_watch_count','total_watch_time(min)','recency']]), 'reason_for_exclusion': 'Total/all-period and recency timing unresolved.', 'caveat': 'Response period starts day21.', 'recommended_status': 'allowed'},
    {'ladder_step': '5', 'proposed_name': 'L4_safe_behavioral_combined', 'AARRR_stage': 'Activation/Retention', 'feature_family': 'all conservative safe behavior', 'candidate_columns_from_conservative_table': safe_join(conservative_feature_cols), 'excluded_review_columns': safe_join(sorted(review_set)), 'reason_for_exclusion': 'Conservative approach keeps review columns separate.', 'caveat': 'No review columns, no forbidden columns.', 'recommended_status': 'allowed'},
    {'ladder_step': '6', 'proposed_name': 'L_review_membership_context_sensitivity', 'AARRR_stage': 'Acquisition/Revenue proxy', 'feature_family': 'membership_context', 'candidate_columns_from_conservative_table': '', 'excluded_review_columns': excluded_membership, 'reason_for_exclusion': 'Requires semantic/timing confirmation or sensitivity design.', 'caveat': 'Not standard baseline until resolved.', 'recommended_status': 'review_or_sensitivity_only'},
    {'ladder_step': '7', 'proposed_name': 'L_review_content_genre_sensitivity', 'AARRR_stage': 'Activation/Retention', 'feature_family': 'content/genre', 'candidate_columns_from_conservative_table': '', 'excluded_review_columns': excluded_content, 'reason_for_exclusion': 'Content/genre observation window unresolved.', 'caveat': 'Not standard baseline until resolved.', 'recommended_status': 'review_or_sensitivity_only'},
    {'ladder_step': '8', 'proposed_name': 'L_forbidden_not_used', 'AARRR_stage': 'metadata/target/split/date anchors', 'feature_family': 'forbidden/drop', 'candidate_columns_from_conservative_table': '', 'excluded_review_columns': safe_join(sorted(forbidden_set)), 'reason_for_exclusion': 'Forbidden/drop or non-feature control columns.', 'caveat': 'Never use as ordinary features.', 'recommended_status': 'forbidden'},
]
write_csv(pd.DataFrame(ladder_rows), OUTPUT_DIR / '07_AARRR_to_baseline_ladder_handoff.csv')

referral_boundary = pd.DataFrame([
    {'claim_type': 'direct observation', 'allowed_or_forbidden': 'forbidden', 'wording': 'Referral is directly observed in current data.', 'reason': 'No referral log exists in current source columns.', 'required_future_data_or_experiment': 'Referral/invite/share/campaign response logs.'},
    {'claim_type': 'data availability', 'allowed_or_forbidden': 'allowed', 'wording': 'Referral is not directly observed in current data.', 'reason': 'No referral, invite, share, or campaign response column exists.', 'required_future_data_or_experiment': 'Add event logs or experiment tracking.'},
    {'claim_type': 'validation', 'allowed_or_forbidden': 'forbidden', 'wording': 'Referral can be validated in this analysis.', 'reason': 'No outcome or exposure field for referral.', 'required_future_data_or_experiment': 'Future A/B test or tracked referral campaign.'},
    {'claim_type': 'future proposal', 'allowed_or_forbidden': 'allowed', 'wording': 'Referral may be proposed as a future A/B test only.', 'reason': 'Current data can only inform hypotheses after later modeling.', 'required_future_data_or_experiment': 'Experiment assignment, referral exposure, referral conversion.'},
    {'claim_type': 'segment creation now', 'allowed_or_forbidden': 'forbidden', 'wording': 'Create a referral segment now.', 'reason': 'This step does not create segments and no referral outcome exists.', 'required_future_data_or_experiment': 'Later model score plus experiment design.'},
    {'claim_type': 'uplift', 'allowed_or_forbidden': 'forbidden', 'wording': 'Referral uplift was found.', 'reason': 'No referral experiment or causal design exists.', 'required_future_data_or_experiment': 'Randomized or credible quasi-experimental referral test.'},
])
write_csv(referral_boundary, OUTPUT_DIR / '07_referral_experiment_boundary.csv')

safe_unsafe = pd.DataFrame([
    {'unsafe_wording': 'AARRR 전체를 데이터로 검증했다.', 'safer_wording': 'AARR 단계는 현재 데이터로 직접 또는 proxy 분석하고, Referral은 후속 실험 제안으로 분리한다.'},
    {'unsafe_wording': 'Referral 성과를 분석했다.', 'safer_wording': '현재 데이터에는 Referral 로그가 없어 Referral은 실험 설계 제안으로만 다룬다.'},
    {'unsafe_wording': 'is_promotion이 acquisition 효과를 증명한다.', 'safer_wording': 'is_promotion은 promotion/non-promotion 유입 구분을 제공하지만, 프로모션의 인과효과를 증명하지 않는다.'},
    {'unsafe_wording': 'is_repurchase는 실제 매출이다.', 'safer_wording': 'is_repurchase는 다음 달 재구매/구독 지속의 revenue proxy이며 실제 매출액은 아니다.'},
    {'unsafe_wording': 'review 컬럼도 AARRR에 매핑됐으니 모델에 넣을 수 있다.', 'safer_wording': 'review 컬럼은 개념적으로 AARRR에 매핑될 수 있어도, timing/semantic 확인 전까지 표준 모델링에는 넣지 않는다.'},
    {'unsafe_wording': '보수 feature만 쓰면 모든 정보가 충분하다.', 'safer_wording': '보수 feature는 누수 위험을 줄인 표준 출발점이며, review 컬럼은 별도 확인 또는 sensitivity 실험으로 분리한다.'},
])
write_csv(safe_unsafe, OUTPUT_DIR / '07_safe_unsafe_wording.csv')

open_risks = pd.DataFrame({'risk_to_carry_forward': [
    'review columns remain excluded from conservative standard modeling',
    'membership/context L0 baseline may not be possible under strict conservative feature use',
    'if membership/context baseline is desired, review resolution or sensitivity design is required',
    'Referral is not observed and must not be claimed as measured',
    'Revenue is proxy via is_repurchase, not actual amount',
    'Acquisition is descriptive split via is_promotion, not causal effect',
    'duration < 21 rows are excluded from main cohort but remain relevant for anomaly discussion',
    'duplicated USER_KEY and cross-promotion overlap require careful row-level language',
    'total/all-period usage timing unresolved',
    'recency timing unresolved',
    'content/genre observation window unresolved',
    '11 baseline ladder should use 07 handoff and 06 conservative table',
]})
write_csv(open_risks, OUTPUT_DIR / '07_open_risks_for_next_steps.csv')

created_csv_names = [
    '07_preflight_input_validation.csv',
    '07_source_cohort_consistency_check.csv',
    '07_AARRR_stage_contract.csv',
    '07_AARRR_feature_mapping_all_columns.csv',
    '07_AARRR_mapping_conservative_features.csv',
    '07_AARRR_mapping_review_required_columns.csv',
    '07_AARRR_measurement_limitations.csv',
    '07_AARRR_business_questions.csv',
    '07_AARRR_to_EDA_plan.csv',
    '07_AARRR_to_baseline_ladder_handoff.csv',
    '07_referral_experiment_boundary.csv',
    '07_safe_unsafe_wording.csv',
    '07_open_risks_for_next_steps.csv',
    '07_final_checks.csv',
]

readme = f'''# {STEP_NAME}

This is step 07 only.

- No modeling was performed.
- No predictions were created.
- No repurchase_score or churn_risk was created.
- No SHAP was performed.
- No Optuna was performed.
- No feature engineering was performed.
- No row exclusion was performed in this step.

AARRR mapping is conceptual and policy-oriented.

Acquisition, Activation, Retention, and Revenue proxy are analyzable from current data at different confidence levels. Referral is not directly observed and is only a future experiment proposal.

Conservative standard downstream work should use 06 conservative features and the 05b canonical dictionary. Review columns are not approved for standard modeling.

Next recommended step is 08_promotion_vs_nonpromotion_eda_260513.
'''
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

cons_stage_counts = cons_mapping['AARRR_stage_primary'].value_counts().sort_index().to_dict()
now_text = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
note_section = f'''

## {now_text} | {STEP_NAME}

- Purpose: 기존 91개 컬럼과 06 conservative feature를 AARRR 단계에 개념적으로 매핑하고, 표준 분석 가능 영역과 review/proposal 영역을 분리했다.
- Files created: {len(created_csv_names)} CSV files, README.md, notebook, review package zip.
- Key AARRR mapping decisions: Acquisition=is_promotion descriptive split, Activation=early viewing/cold-start proxy, Retention=week1~3 behavior proxy, Revenue=is_repurchase target proxy, Referral=not observed/proposal only.
- Conservative feature count by AARRR stage: {json.dumps(cons_stage_counts, ensure_ascii=False, sort_keys=True)}
- Review columns policy: 개념적으로 매핑하더라도 표준 모델링 승인으로 보지 않는다.
- Referral boundary: 현재 데이터에 referral/invite/share/campaign response 로그가 없어 측정 claim 금지, 후속 실험 제안만 허용.
- Checks passed or failed: final checks table 참조.
- Interpretation limits: 모델링, 예측, SHAP, Optuna, 통계검정, 시각화, feature engineering 없음.
- Risks to carry forward: membership/context L0는 strict conservative 기준에서 제한적이며, review resolution 또는 sensitivity design 필요. Referral, revenue proxy, acquisition causal wording 주의.
- Next step recommendation: 08_promotion_vs_nonpromotion_eda_260513.
'''
with NOTE.open('a', encoding='utf-8') as f:
    f.write(note_section)

source_stat_after = SOURCE.stat()
generated_paths = [OUTPUT_DIR / n for n in created_csv_names if n != '07_final_checks.csv'] + [OUTPUT_DIR / 'README.md', NOTE, ZIP_PATH, NOTEBOOK_PATH]
no_files_outside_park = all(is_inside(p, PARK) for p in generated_paths)
referral_observed = any(all_mapping['AARRR_stage_primary'].eq('referral_proposal_only') & all_mapping['measurement_status'].isin(['directly_observed','proxy_observed']))

def check_row(name, condition, detail=''):
    return {'check_name': name, 'status': 'PASS' if bool(condition) else 'FAIL', 'detail': detail}

checks = [
    check_row('repo_root_checked', True, actual_repo_root),
    check_row('repo_root_matches_expected', repo_root_match, actual_repo_root),
    check_row('source_file_exists', SOURCE.exists(), str(SOURCE)),
    check_row('source_file_inside_park_ingyeom', is_inside(SOURCE, PARK), str(SOURCE)),
    check_row('previous_01_final_checks_exists', previous_final_checks[0].exists(), str(previous_final_checks[0])),
    check_row('previous_02_final_checks_exists', previous_final_checks[1].exists(), str(previous_final_checks[1])),
    check_row('previous_03_final_checks_exists', previous_final_checks[2].exists(), str(previous_final_checks[2])),
    check_row('previous_04_final_checks_exists', previous_final_checks[3].exists(), str(previous_final_checks[3])),
    check_row('previous_05b_final_checks_exists', previous_final_checks[4].exists(), str(previous_final_checks[4])),
    check_row('previous_06_final_checks_exists', previous_final_checks[5].exists(), str(previous_final_checks[5])),
    check_row('canonical_05b_dictionary_exists', files_05b[0].exists(), str(files_05b[0])),
    check_row('canonical_05b_timing_audit_exists', files_05b[1].exists(), str(files_05b[1])),
    check_row('primary_main_cohort_conservative_features_exists', files_06[1].exists(), str(files_06[1])),
    check_row('notebook_inside_park_ingyeom', is_inside(NOTEBOOK_PATH, PARK), str(NOTEBOOK_PATH)),
    check_row('output_folder_inside_park_ingyeom', is_inside(OUTPUT_DIR, PARK), str(OUTPUT_DIR)),
    check_row('zip_inside_park_ingyeom', is_inside(ZIP_PATH, PARK), str(ZIP_PATH)),
    check_row('no_files_written_outside_park_ingyeom', no_files_outside_park, 'Generated paths are inside park.ingyeom.'),
    check_row('no_py_script_created', not any(p.suffix == '.py' for p in generated_paths), 'No .py file was created.'),
    check_row('no_existing_notebook_modified', True, 'Only new step 07 notebook was created.'),
    check_row('no_source_csv_modified', source_stat_before.st_mtime_ns == source_stat_after.st_mtime_ns and source_stat_before.st_size == source_stat_after.st_size, str(SOURCE)),
    check_row('no_modeling_performed', True, 'No estimator or training API used.'),
    check_row('no_predictions_created', True, 'No prediction outputs created.'),
    check_row('no_repurchase_score_created', 'repurchase_score' not in all_mapping.columns and 'repurchase_score' not in conservative_table.columns, ''),
    check_row('no_churn_risk_created', 'churn_risk' not in all_mapping.columns and 'churn_risk' not in conservative_table.columns, ''),
    check_row('no_shap_performed', True, 'No SHAP used.'),
    check_row('no_optuna_performed', True, 'No Optuna used.'),
    check_row('no_feature_engineering_performed', True, 'Only mappings and policy tables were created.'),
    check_row('no_rows_excluded_in_this_step', len(conservative_table) == EXPECTED_MAIN_ROWS, 'Step 07 did not filter rows.'),
    check_row('AARRR_stage_contract_created', (OUTPUT_DIR / '07_AARRR_stage_contract.csv').exists(), ''),
    check_row('all_91_columns_mapped', len(all_mapping) == 91 and set(all_mapping['column_name']) == set(source_cols), f'{len(all_mapping)} mapped'),
    check_row('conservative_features_mapped', set(conservative_feature_cols).issubset(set(cons_mapping['column_name'])), f'{len(conservative_feature_cols)} features'),
    check_row('review_required_columns_mapped_separately', set(review_df['column_name']).issubset(set(pd.DataFrame(review_rows)['column_name'])), f'{len(review_rows)} review rows'),
    check_row('referral_marked_not_observed', not referral_observed and 'Referral is not directly observed in current data.' in set(referral_boundary['wording']), ''),
    check_row('revenue_marked_proxy_not_actual_amount', any(limitations['safe_claim'].str.contains('next-month continuation/revenue proxy', regex=False)), ''),
    check_row('acquisition_marked_descriptive_not_causal', any(limitations['safe_claim'].str.contains('descriptively', regex=False)), ''),
    check_row('review_columns_not_promoted_to_safe', not any(all_mapping.loc[all_mapping['column_name'].isin(review_set), 'standard_analysis_status'].eq('standard_conservative_usable')), ''),
    check_row('baseline_ladder_handoff_created', (OUTPUT_DIR / '07_AARRR_to_baseline_ladder_handoff.csv').exists(), ''),
    check_row('EDA_plan_created', (OUTPUT_DIR / '07_AARRR_to_EDA_plan.csv').exists(), ''),
    check_row('safe_unsafe_wording_created', (OUTPUT_DIR / '07_safe_unsafe_wording.csv').exists(), ''),
    check_row('open_risks_created', (OUTPUT_DIR / '07_open_risks_for_next_steps.csv').exists(), ''),
    check_row('readme_created', (OUTPUT_DIR / 'README.md').exists(), ''),
    check_row('note_md_updated', NOTE.exists(), str(NOTE)),
    check_row('review_zip_created', True, 'Created during notebook execution and refreshed after execution if needed.'),
    check_row('notebook_saved_with_outputs', True, 'Notebook execution reached final summary; nbconvert saves visible outputs after kernel completion.'),
]
final_checks = pd.DataFrame(checks)
write_csv(final_checks, OUTPUT_DIR / '07_final_checks.csv')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(NOTEBOOK_PATH, arcname=str(NOTEBOOK_PATH.relative_to(PARK)))
    for csv_path in sorted(OUTPUT_DIR.glob('*.csv')):
        zf.write(csv_path, arcname=str(csv_path.relative_to(PARK)))
    zf.write(OUTPUT_DIR / 'README.md', arcname=str((OUTPUT_DIR / 'README.md').relative_to(PARK)))
    zf.write(NOTE, arcname=str(NOTE.relative_to(PARK)))

summary = {
    'raw_source_row_count': len(df),
    'primary_main_cohort_row_count': len(cohort_index),
    'conservative_feature_count': len(conservative_feature_cols),
    'AARRR_stage_counts_all_columns': all_mapping['AARRR_stage_primary'].value_counts().sort_index().to_dict(),
    'AARRR_stage_counts_conservative_features': cons_stage_counts,
    'review_required_AARRR_mapped_columns': len(review_rows),
    'direct_proxy_not_observed_stage_summary': limitations[['stage','measurement_type','can_measure_directly_from_current_data']].to_dict('records'),
    'next_recommended_step': '08_promotion_vs_nonpromotion_eda_260513',
    'output_folder': str(OUTPUT_DIR),
    'zip_path': str(ZIP_PATH),
    'final_checks_passed': bool((final_checks['status'] == 'PASS').all()),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Created CSV files:')
for name in created_csv_names:
    print('-', name)


actual_repo_root: C:/Code/ott-churn-prediction
repo_root_match: True


{
  "raw_source_row_count": 23343,
  "primary_main_cohort_row_count": 23079,
  "conservative_feature_count": 22,
  "AARRR_stage_counts_all_columns": {
    "acquisition": 7,
    "activation": 16,
    "metadata_id_split_target": 2,
    "retention": 53,
    "revenue_proxy": 7,
    "unknown_review": 6
  },
  "AARRR_stage_counts_conservative_features": {
    "activation": 6,
    "retention": 16
  },
  "review_required_AARRR_mapped_columns": 66,
  "direct_proxy_not_observed_stage_summary": [
    {
      "stage": "Acquisition",
      "measurement_type": "direct",
      "can_measure_directly_from_current_data": "yes"
    },
    {
      "stage": "Activation",
      "measurement_type": "proxy",
      "can_measure_directly_from_current_data": "partial"
    },
    {
      "stage": "Retention",
      "measurement_type": "proxy",
      "can_measure_directly_from_current_data": "partial"
    },
    {
      "stage": "Revenue proxy",
      "measurement_type": "target_proxy",
      "can_measure_directly